# CNN + BiLSTM + Attention — AIRTLab và webcam trên Kaggle

Notebook độc lập, checkpoint tương thích `src/models/cnn_lstm.py` và backend localhost.

## 1. Chạy ngay với Input AIRTLab của bạn

Cell cấu hình đã đặt đúng `DATA_ROOT` bạn cung cấp, `DATA_MODE='airtlab'`.
Giữ `INIT_CHECKPOINT=None` để train từ backbone ImageNet; bật GPU và Internet khi cần tải weights,
sau đó Restart & Run All. Không cần tự tạo `manifest.csv` hoặc dùng ba CSV hành động có sẵn.

Notebook đọc cấu trúc `non-violent/cam1`, `non-violent/cam2`, `violent/cam1`, `violent/cam2`.
Tên video số giống nhau trong cùng lớp được ghép thành một nhóm, ví dụ
`non-violent/cam1/1.mp4` và `non-violent/cam2/1.mp4`.
Các nhóm được chia gần 70/15/15 theo lớp với seed cố định; hai góc của cùng tình huống không bị chia ra.
Đây là chia theo **tình huống**, chưa phải chia độc lập theo người vì không có ID diễn viên/buổi quay.
Nguồn cấu trúc: [AIRTLab chính thức](https://github.com/airtlab/A-Dataset-for-Automatic-Violence-Detection-in-Videos).

Video ngắn hơn khoảng lấy mẫu 2,25 giây sẽ bị loại cùng góc camera còn lại và ghi vào
`excluded_airtlab.csv`. Video hỏng sẽ báo lỗi, không tự thay bằng ảnh đen.
Manifest được ghi trong `/kaggle/working/webcam_cnn_lstm/manifest.csv`; Input Kaggle chỉ được đọc.
Đường dẫn video trong manifest tương đối với `DATA_ROOT`, không phải thư mục Output.

**AIRTLab được giữ nhãn nguồn `external`.** Checkpoint/ngưỡng được chọn bằng validation AIRTLab;
báo cáo test là kết quả AIRTLab, không phải độ chính xác webcam. Model có thể nạp vào web,
nhưng cần test và fine-tune trên webcam thật trước khi kết luận chất lượng triển khai.

## Khi đã có dữ liệu webcam tự quay

Đổi `DATA_MODE='manifest'`, đặt `MANIFEST_PATH` tới CSV tự chuẩn bị.
Trong chế độ này đường dẫn video tương đối với thư mục chứa CSV; `DATA_ROOT` không được dùng.

```csv
path,label,group_id,split,domain
videos/session01_normal.mp4,0,session01,train,webcam
videos/session01_fight.mp4,1,session01,train,webcam
videos/session02_normal.mp4,0,session02,val,webcam
videos/session02_fight.mp4,1,session02,val,webcam
videos/session03_normal.mp4,0,session03,test,webcam
videos/session03_fight.mp4,1,session03,test,webcam
```

Đây chỉ là mẫu định dạng. Cần nhiều người/buổi quay, đủ 2 lớp cho mỗi split webcam.
Gộp các clip cùng người/buổi quay/video nguồn cùng nhóm trước khi chia; không dùng lại test để train.
Có thể thêm dữ liệu ngoài với `domain=external`, nhưng giữ nguyên split của dữ liệu đã dùng trước đó.
Gán nhãn nhất quán trong mỗi video, cắt đoạn có trạng thái khác nhau trước. Video cần FPS đúng thực tế,
dài ít nhất 2,25 giây. Các mẫu bình thường dễ nhầm gồm vẫy tay, ôm, tập thể dục và đùa giỡn.

In [25]:
"""Train one clean pipeline: ResNet50 + BiLSTM + temporal Attention.

Kaggle: change DATA_ROOT and optionally INIT_CHECKPOINT, then Run All.
"""
import copy, random
from pathlib import Path
import cv2, numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights



torch: 2.10.0+cu128 | CUDA: True


In [3]:
!echo "===== DATASET ====="
!find /kaggle/input -type d -name "violence-detection-dataset"

!echo ""
!echo "===== CHECKPOINT ====="
!find /kaggle/input -type f -name "best.pt"

===== DATASET =====
/kaggle/input/datasets/bohong111/datawebcam/A-Dataset-for-Automatic-Violence-Detection-in-Videos-master/violence-detection-dataset

===== CHECKPOINT =====


In [27]:
def build_airtlab_manifest(data_root, output_csv, seed):
    """Group both cameras of each class/clip ID before stratified splitting."""
    data_root, output_csv = Path(data_root).resolve(), Path(output_csv)
    rows = []
    for class_name, label in [('non-violent', 0), ('violent', 1)]:
        for camera in ('cam1', 'cam2'):
            folder = data_root / class_name / camera
            if not folder.is_dir():
                raise FileNotFoundError(f'Thiếu thư mục AIRTLab: {folder}')
            videos = sorted(p for p in folder.iterdir()
                            if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'})
            if not videos:
                raise ValueError(f'Không có video trong {folder}')
            for video in videos:
                if not video.stem.isdecimal():
                    raise ValueError(f'AIRTLab gốc dùng tên số, không tự đoán cặp camera: {video.name}')
                cap = cv2.VideoCapture(str(video))
                try:
                    fps = float(cap.get(cv2.CAP_PROP_FPS))
                    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                    ok, _ = cap.read()
                    if not cap.isOpened() or not ok or not np.isfinite(fps) or fps <= 0 or count <= 0:
                        raise ValueError(f'Không đọc được video/FPS: {video}. Kiểm tra Input có video thật.')
                finally:
                    cap.release()
                rows.append({'path': video.relative_to(data_root).as_posix(), 'label': label,
                             'group_id': f'airtlab/{class_name}/{int(video.stem)}',
                             'domain': 'external', 'camera': camera, 'last_time': (count - 1) / fps})
    df = pd.DataFrame(rows)
    for group, subset in df.groupby('group_id'):
        if len(subset) != 2 or set(subset.camera) != {'cam1', 'cam2'}:
            raise ValueError(f'Cặp camera thiếu/trùng: {group}')

    # Có clip AIRTLab chỉ dài 2s: loại CẢ NHÓM nếu một góc không đủ 2.25s.
    # Không kéo giãn thời gian hoặc lặp frame để che việc thiếu chuyển động.
    short_groups = set(df.loc[df.last_time + 1e-6 < CLIP_SPAN_SECONDS, 'group_id'])
    excluded = df[df.group_id.isin(short_groups)].copy()
    excluded['reason'] = 'camera_pair_shorter_than_required_clip_span'
    df = df[~df.group_id.isin(short_groups)].copy()
    rng = np.random.default_rng(seed)
    assignments = {}
    for label in (0, 1):
        groups = sorted(df.loc[df.label == label, 'group_id'].unique())
        if len(groups) < 3:
            raise ValueError(f'Lớp {label} cần ít nhất 3 cặp camera hợp lệ để chia train/val/test.')
        groups = list(rng.permutation(groups))
        n_val = max(1, round(len(groups) * 0.15))
        n_test = max(1, round(len(groups) * 0.15))
        n_train = len(groups) - n_val - n_test
        for split, selected in [('train', groups[:n_train]),
                                ('val', groups[n_train:n_train + n_val]),
                                ('test', groups[n_train + n_val:])]:
            assignments.update({group: split for group in selected})
    df['split'] = df.group_id.map(assignments)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    excluded.to_csv(output_csv.parent / 'excluded_airtlab.csv', index=False)
    df[['path', 'label', 'group_id', 'split', 'domain']].to_csv(output_csv, index=False)
    print(f'AIRTLab: {len(df)} video, {df.group_id.nunique()} cặp camera; loại {len(excluded)} video ngắn/cùng cặp.')
    print('Manifest:', output_csv)
    return df


def validate_manifest(csv_path, data_root=None):
    csv_path = Path(csv_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy manifest: {csv_path}. '
                           'Với AIRTLab, chạy trọn cell tạo và kiểm tra manifest; '
                           'với chế độ manifest, kiểm tra đường dẫn CSV đã chuẩn bị.')
    df = pd.read_csv(csv_path, dtype={'path': str, 'group_id': str, 'split': str, 'domain': str})
    required = {'path', 'label', 'group_id', 'split', 'domain'}
    if not required.issubset(df.columns) or df.empty:
        raise ValueError(f'Manifest phải có dữ liệu và các cột {sorted(required)}')
    if df[list(required)].isna().any().any():
        raise ValueError('Manifest có ô trống.')
    for col in ('path', 'group_id', 'split', 'domain'):
        df[col] = df[col].str.strip()
        if df[col].eq('').any():
            raise ValueError(f'Cột {col} có giá trị trống.')
    if not df.label.isin([0, 1]).all():
        raise ValueError('label chỉ nhận 0 hoặc 1.')
    df['label'] = df.label.astype(int)
    if not df.split.isin(['train', 'val', 'test']).all():
        raise ValueError('split chỉ nhận train/val/test.')
    if not df.domain.isin(['webcam', 'external']).all():
        raise ValueError('domain chỉ nhận webcam/external.')
    base = Path(data_root).resolve() if data_root is not None else csv_path.parent.resolve()
    resolved = []
    for value in df.path:
        rel = Path(value)
        path = (base / rel).resolve()
        if rel.is_absolute() or not path.is_relative_to(base):
            raise ValueError(f'path phải tương đối và nằm trong dataset: {value}')
        if not path.is_file():
            raise FileNotFoundError(path)
        resolved.append(str(path))
    df['resolved_path'] = resolved
    if df.resolved_path.duplicated().any():
        raise ValueError('Một file xuất hiện nhiều lần trong manifest.')
    if (df.groupby('group_id').split.nunique() > 1).any():
        raise ValueError('Rò rỉ dữ liệu: cùng group_id xuất hiện ở nhiều split.')
    hashes = []
    for path in tqdm(df.resolved_path, desc='Kiểm tra file trùng'):
        digest = hashlib.sha256()
        with open(path, 'rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        hashes.append(digest.hexdigest())
    df['sha256'] = hashes
    if df.sha256.duplicated().any():
        raise ValueError('Có video trùng nội dung; bỏ bản sao và kiểm tra lại split/nhãn.')
    for split in ('train', 'val', 'test'):
        target = df[(df.split == split) & (df.domain == EVAL_DOMAIN)]
        if set(target.label) != {0, 1}:
            raise ValueError(f'{split} {EVAL_DOMAIN} cần đủ 2 lớp để train/đánh giá đúng mục tiêu.')
        if target.group_id.nunique() < 3:
            warnings.warn(f'{split} {EVAL_DOMAIN} có dưới 3 nhóm; số liệu có thể dao động mạnh.')
    return df

def build_rwf_manifest(data_root, output_csv, seed):
    rows = []
    for split in ('train', 'val'):
        for folder_name, label in (('NonFight', 0), ('Fight', 1)):
            folder = Path(data_root) / split / folder_name
            if not folder.is_dir():
                raise FileNotFoundError(f'Thiếu thư mục RWF-2000: {folder}')
            for video in sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in {'.mp4', '.avi', '.mov', '.mkv'}):
                rows.append({'path': str(video.relative_to(data_root)).replace('\\', '/'), 'label': label, 'group_id': video.stem, 'source_split': split, 'domain': 'external'})
    df = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Giữ split gốc của RWF; val được dùng làm validation/test nội bộ khi cần.
    df['split'] = df['source_split'].map({'train': 'train', 'val': 'test'})
    val_rows = df[df.split == 'test'].groupby('label', group_keys=False).apply(lambda x: x.sample(frac=0.5, random_state=seed)).index
    df.loc[val_rows, 'split'] = 'val'
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df[['path', 'label', 'group_id', 'split', 'domain']].to_csv(output_csv, index=False)
    print(f'RWF-2000: {len(df)} video')
    return df


def prepare_manifest():
    # Cùng cell định nghĩa builder và validator: không phụ thuộc cell tạo CSV riêng.
    # Giữ manifest đã tồn tại để không tự thay đổi split khi chạy lại cell.
    if DATA_MODE == 'rwf2000' and not Path(MANIFEST_PATH).is_file():
        print('Đang tạo manifest từ RWF-2000:', DATA_ROOT)
        build_rwf_manifest(DATA_ROOT, MANIFEST_PATH, CFG['seed'])
    if DATA_MODE == 'airtlab' and not Path(MANIFEST_PATH).is_file():
        print('Chưa có manifest; đang tạo từ AIRTLab:', DATA_ROOT)
        build_airtlab_manifest(DATA_ROOT, MANIFEST_PATH, CFG['seed'])
    if DATA_MODE == 'rwf2000':
        print('Đánh giá trên RWF-2000 (external), chưa phải độ chính xác webcam thực tế.')
    elif DATA_MODE == 'airtlab':
        print('Đánh giá trên AIRTLab (external), chưa phải độ chính xác webcam thực tế.')
    return validate_manifest(MANIFEST_PATH, DATA_ROOT if DATA_MODE in {'rwf2000', 'airtlab'} else None)


manifest = prepare_manifest()
print(manifest.groupby(['split', 'domain', 'label']).size().rename('videos').to_string())
manifest.to_csv(OUT_DIR / 'manifest_resolved.csv', index=False)

Đánh giá trên RWF-2000 (external), chưa phải độ chính xác webcam thực tế.


Kiểm tra file trùng:   0%|          | 0/1989 [00:00<?, ?it/s]

split  domain    label
test   external  0         97
                 1        100
train  external  0        798
                 1        797
val    external  0         97
                 1        100


## 3. Lấy frame theo thời gian và kiểm tra ảnh đầu vào

Train: cửa sổ ngẫu nhiên trong video đã gán nhãn, 16 frame cách nhau 150 ms.
Validation/test: duyệt cửa sổ cố định bước 1,1 giây, thêm cửa sổ cuối video.
Đánh giá **theo cửa sổ**, không lấy trung bình toàn video để che mất hành động ngắn.
Video ngắn hơn cửa sổ hoặc frame lỗi sẽ báo lỗi, không tạo ảnh đen thay thế.

Mô phỏng đường truyền webcam bằng resize cạnh dài tối đa 384 và JPEG quality 70
(xấp xỉ browser JPEG 0.7; bộ mã hóa trình duyệt có thể khác).
Sau đó dùng Resize(size + 32), CenterCrop và chuẩn hóa ImageNet giống web.
Augmentation train dùng cùng thông số cho cả clip để giữ chuyển động nhất quán.

In [28]:
from pathlib import Path
import random
import numpy as np
import torch

# =========================
# PATH
# =========================

DATA_ROOT = Path(
    "/kaggle/input/datasets/bohong111/datawebcam/A-Dataset-for-Automatic-Violence-Detection-in-Videos-master/violence-detection-dataset"
)

INIT_CHECKPOINT = Path(
    "/kaggle/input/models/hoangbao123/modelv1/pytorch/default/1/best.pt"
)

OUT_DIR = Path(
    "/kaggle/working/resnet50_bilstm_attention"
)

# =========================
# CONFIG
# =========================

SEED = 42
T = 16
SIZE = 224
BATCH = 2
EPOCHS = 20
PATIENCE = 5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EXTS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpg", ".mpeg", ".webm"
}

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)


# =========================
# CHECK
# =========================

print("===== KIỂM TRA CẤU HÌNH =====")

print("Device:", DEVICE)

print("\nDataset:")
print(DATA_ROOT)
print("Dataset tồn tại:", DATA_ROOT.exists())

print("\nCheckpoint:")
print(INIT_CHECKPOINT)
print("Checkpoint tồn tại:", INIT_CHECKPOINT.exists())


if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy dataset:\n{DATA_ROOT}"
    )

if not INIT_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy model:\n{INIT_CHECKPOINT}"
    )


print("\nOK - Dataset đã tìm thấy.")
print("OK - Model đã tìm thấy.")
print("Có thể chạy cell tiếp theo.")

Kiểm tra video:   0%|          | 0/1989 [00:00<?, ?it/s]

In [29]:
def videos(folder):
    return sorted(
        p for p in Path(folder).rglob("*")
        if p.is_file() and p.suffix.lower() in EXTS
    )


def collect():
    # Dataset hiện tại:
    #
    # violence-detection-dataset/
    # ├── non-violent/
    # │   ├── cam1/
    # │   └── cam2/
    # └── violent/
    #     ├── cam1/
    #     └── cam2/

    nonviolent = videos(
        DATA_ROOT / "non-violent"
    )

    violent = videos(
        DATA_ROOT / "violent"
    )

    print("===== DATASET =====")
    print("Non-violent:", len(nonviolent))
    print("Violent    :", len(violent))

    # Kiểm tra dữ liệu
    if len(nonviolent) == 0 or len(violent) == 0:
        raise FileNotFoundError(
            f"Không tìm thấy video violent/non-violent trong:\n{DATA_ROOT}"
        )

    # =========================
    # LABEL
    # =========================
    # 0 = non-violent
    # 1 = violent

    paths = (
        [str(p) for p in nonviolent]
        +
        [str(p) for p in violent]
    )

    labels = (
        [0] * len(nonviolent)
        +
        [1] * len(violent)
    )

    # =========================
    # TRAIN 70%
    # TEMP 30%
    # =========================

    train_p, temp_p, train_y, temp_y = train_test_split(
        paths,
        labels,
        test_size=0.30,
        random_state=SEED,
        stratify=labels
    )

    # =========================
    # TEMP -> VAL 15% + TEST 15%
    # =========================

    val_p, test_p, val_y, test_y = train_test_split(
        temp_p,
        temp_y,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_y
    )

    # =========================
    # INFO
    # =========================

    print("\n===== SPLIT =====")
    print("Train:", len(train_p))
    print("Val  :", len(val_p))
    print("Test :", len(test_p))

    print("\nTrain:")
    print("  Non-violent:", train_y.count(0))
    print("  Violent    :", train_y.count(1))

    print("\nVal:")
    print("  Non-violent:", val_y.count(0))
    print("  Violent    :", val_y.count(1))

    print("\nTest:")
    print("  Non-violent:", test_y.count(0))
    print("  Violent    :", test_y.count(1))

    return (
        train_p,
        train_y,
        val_p,
        val_y,
        test_p,
        test_y
    )

Train videos / validation windows / test windows: 1595 788 788


## 4. CNN + BiLSTM + Attention

Các lớp bên dưới được đồng bộ từ source hiện tại để giữ tên và kích thước trọng số.
Mặc định ResNet50; có thể chọn ResNet18 khi train mới để giảm chi phí CPU localhost.
Nếu nạp checkpoint cũ, kiến trúc được lấy từ checkpoint và kiểm tra bằng `strict=True`.

In [30]:
MEAN, STD = (0.485, .456, .406), (.229, .224, .225)
train_tf = transforms.Compose([
    transforms.Resize((SIZE+32, SIZE+32)), transforms.RandomCrop(SIZE),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((SIZE, SIZE)), transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)])



In [31]:
def read_clip(path, train):
    cap = cv2.VideoCapture(str(path)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not cap.isOpened() or n <= 0:
        cap.release(); raise RuntimeError(f"Không đọc được video: {path}")
    # Random contiguous clip only for train; fixed evenly-spaced clip for eval.
    start = random.randint(0, max(0, n-T)) if train else max(0, (n-T)//2)
    frames = []
    for idx in range(start, min(start+T, n)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, bgr = cap.read()
        if ok: frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames: raise RuntimeError(f"Không decode được video: {path}")
    while len(frames) < T: frames.append(frames[-1])
    tf = train_tf if train else eval_tf
    return torch.stack([tf(Image.fromarray(f)) for f in frames[:T]])

class VideoDataset(Dataset):
    def __init__(self, paths, labels, train=False):
        self.paths, self.labels, self.train = paths, labels, train
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return read_clip(self.paths[i], self.train), torch.tensor(self.labels[i], dtype=torch.long)



Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 209MB/s]


Parameters: 40303682


## 5. Huấn luyện và chọn checkpoint

Warmup LSTM/attention, sau đó fine-tune tầng cuối CNN bằng learning rate nhỏ.
AMP giảm bộ nhớ GPU, gradient clipping hạn chế gradient lớn.
Chọn checkpoint bằng macro-F1 validation tại ngưỡng 0.5 trên `EVAL_DOMAIN`:
`external` cho AIRTLab, `webcam` cho chế độ manifest webcam. Không dùng test để chọn model.

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, d):
        super().__init__(); self.score = nn.Sequential(nn.Linear(d,d//2), nn.Tanh(), nn.Linear(d//2,1))
    def forward(self, h):
        a = torch.softmax(self.score(h).squeeze(-1), dim=1)
        return torch.bmm(a.unsqueeze(1), h).squeeze(1), a

class ResNet50BiLSTMAttention(nn.Module):
    def __init__(self):
        super().__init__(); net = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(net.children())[:-1])
        self.lstm = nn.LSTM(2048, 512, 2, batch_first=True, bidirectional=True, dropout=.5)
        self.attention = TemporalAttention(1024)
        self.head = nn.Sequential(nn.Dropout(.5), nn.Linear(1024, 2))
    def forward(self, x):
        b,t,c,h,w = x.shape; z = self.cnn(x.reshape(b*t,c,h,w)).flatten(1)
        z,_ = self.lstm(z.view(b,t,2048)); context, alpha = self.attention(z)
        return self.head(context), alpha



Train 1:   0%|          | 0/798 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/394 [00:00<?, ?it/s]

epoch=01 loss=0.7056 external_val_macro_f1=0.6786


Train 2:   0%|          | 0/798 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/394 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f7da699eb60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f7da699eb60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

epoch=02 loss=0.6251 external_val_macro_f1=0.7121


Train 3:   0%|          | 0/798 [00:00<?, ?it/s]

## 6. Chọn ngưỡng từ validation, rồi đánh giá test

Tối đa macro-F1 trên validation của `EVAL_DOMAIN`, báo cáo precision, recall và tỷ lệ báo nhầm.
Ngưỡng AIRTLab chưa được hiệu chỉnh trên webcam thật. Không dùng test để chỉnh ngưỡng.
Các cửa sổ chồng lấn không phải các mẫu độc lập. Đây chưa phải đo cảnh báo/phút hoặc độ trễ camera.

In [ ]:
def load_old(model, path):
    if not path: return model
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt); mapped = {}
    # Supports project checkpoints (spatial/temporal) and this notebook format.
    for k,v in state.items():
        k = k.replace("spatial.features.", "cnn.").replace("temporal.lstm.", "lstm.")
        k = k.replace("temporal.attn.", "attention.").replace("temporal.classifier.", "head.")
        mapped[k] = v
    missing, unexpected = model.load_state_dict(mapped, strict=False)
    print("Loaded checkpoint:", path, "missing:", len(missing), "unexpected:", len(unexpected))
    return model



In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None; model.train(train); ys=[]; ps=[]; losses=[]
    with (torch.enable_grad() if train else torch.no_grad()):
        for x,y in tqdm(loader, leave=False):
            x,y = x.to(DEVICE), y.to(DEVICE)
            if train: optimizer.zero_grad(set_to_none=True)
            logits,_ = model(x); loss = criterion(logits,y)
            if train: loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); optimizer.step()
            losses.append(loss.item()); ys += y.cpu().tolist(); ps += logits.argmax(1).cpu().tolist()
    return {"loss": float(np.mean(losses)), "accuracy": accuracy_score(ys,ps),
            "f1": f1_score(ys,ps,average="macro",zero_division=0)}



In [ ]:
train_p, train_y, val_p, val_y, test_p, test_y = collect()
loader = lambda p,y,tr=False: DataLoader(VideoDataset(p,y,tr), batch_size=BATCH, shuffle=tr,
                                           num_workers=0, pin_memory=DEVICE.type=="cuda")
train_loader, val_loader, test_loader = loader(train_p,train_y,True), loader(val_p,val_y), loader(test_p,test_y)
model = load_old(ResNet50BiLSTMAttention().to(DEVICE), INIT_CHECKPOINT)
criterion = nn.CrossEntropyLoss(); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
best, stale, best_path = -1.0, 0, OUT_DIR / "best.pt"
print("Device:", DEVICE, "train/val/test:", len(train_p), len(val_p), len(test_p))
for epoch in range(1,EPOCHS+1):
    tr = run_epoch(model, train_loader, criterion, optimizer); va = run_epoch(model,val_loader,criterion)
    print(f"epoch {epoch:02d} train={tr} val={va}")
    if va["f1"] > best:
        best, stale = va["f1"], 0
        torch.save({"model":model.state_dict(), "cfg":{"data":{"classes":["non_violence","violence"],"num_frames":T,"image_size":SIZE},
            "model":{"backbone":"resnet50","pretrained":False,"freeze_backbone":False,"lstm_hidden":512,"lstm_layers":2,"bidirectional":True,"dropout":.5,"use_attention":True,"num_classes":2},
            "infer":{"window_frames":T,"stride":8,"threshold":.6}}, "val":va}, best_path)
        print("saved", best_path)
    else:
        stale += 1
        if stale >= PATIENCE: break
print("TEST:", run_epoch(model,test_loader,criterion)); print("BEST:", best_path)


## 7. Đưa checkpoint về localhost

1. Tải `webcam_cnn_lstm/best.pt` và báo cáo từ Output Kaggle. Giữ một bản checkpoint cũ để so sánh.
2. Chép checkpoint mới vào `checkpoints/best.pt` trong project và **restart** web:

   ```powershell
   python -m uvicorn src.api:app --host 127.0.0.1 --port 8000
   ```

3. Xem `/health`, sau đó mở `/live`. Backend đọc cấu hình và ngưỡng từ checkpoint mới.
   Không cần sửa ngưỡng trong `default.yaml`. Ngưỡng được chọn trên domain ghi trong checkpoint; AIRTLab chưa thay thế việc hiệu chỉnh webcam thật.
4. Frontend hiện lấy 16 frame cách khoảng 150 ms, giảm cạnh dài còn 384 và JPEG quality 0.7;
   notebook mô phỏng thông số này. Chỉ đánh giá sau khi buffer đủ 16 frame (khoảng 2,4 giây).
   Frontend cũ cho phép request từ 8 frame; những dự đoán khởi động đó chưa khớp luồng train.
   Thời gian lấy frame trình duyệt có thể trễ; cần đo lại trên máy sử dụng thật.
5. Thử các buổi quay mới, ghi nhận báo nhầm và bỏ sót. Không dùng lại test này để train rồi tiếp tục
   báo cáo nó như test độc lập. Muốn fine-tune tiếp phải giữ một test khác chưa sử dụng.

**Phạm vi tương thích:** checkpoint nạp được vào backend hiện tại; kiến trúc, RGB, resize/crop và
chuẩn hóa giống source. Notebook này thay đổi train/evaluation, không thay source web.
`src.infer` ở đường **upload** vẫn dùng cửa sổ frame liên tiếp; công cụ đánh giá local cũ đã được gỡ.
Đường upload chưa tái hiện cách lấy mẫu theo thời gian của notebook. Không so trực tiếp các số liệu đó
với `metrics.json`; dùng notebook để đánh giá video webcam theo đúng protocol này.

Khung người HOG trên web không phải định vị người đang bạo lực. Phần trăm là xác suất cho cả clip.
Nếu kết quả vẫn kém, xem các cửa sổ dự đoán sai trong CSV, kiểm tra crop/FPS/nhãn và bổ sung dữ liệu
webcam tương ứng trước khi tăng kích thước mô hình.